# AGGREGATION

The aggregation works with the Split - Apply - Combine mechanism.

In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

In [13]:
df = pd.read_csv('../1_Exploratory_DA_Use_This/pandas_folder/data/Large_Countries/large_countries_2015.csv', index_col=0)
df

,population,fertility,continent
Bangladesh,1.609956e+08,2.12,Asia
Brazil,2.078475e+08,1.78,South America
China,1.376049e+09,1.57,Asia
India,1.311051e+09,2.43,Asia
Indonesia,2.575638e+08,2.28,Asia
Japan,1.265735e+08,1.45,Asia
Mexico,1.270172e+08,2.13,North America
Nigeria,1.822020e+08,5.89,Africa
Pakistan,1.889249e+08,3.04,Asia
Philippines,1.006994e+08,2.98,Asia


Let's transform the unit of the population to millions.

In [14]:
df.population = round(df.population/1000000, 1)


In [15]:
df

,population,fertility,continent
Bangladesh,161.0,2.12,Asia
Brazil,207.8,1.78,South America
China,1376.0,1.57,Asia
India,1311.1,2.43,Asia
Indonesia,257.6,2.28,Asia
Japan,126.6,1.45,Asia
Mexico,127.0,2.13,North America
Nigeria,182.2,5.89,Africa
Pakistan,188.9,3.04,Asia
Philippines,100.7,2.98,Asia


Today we are going to talk about 3 different but related topics.

1. Aggreate functions in pandas
2. The groupby method of a pandas.DataFrame
3. Applying functions or transformations to pandas.Series

# Section I: Aggregate functions in pandas

**Aggregate function**: takes multiple rows as input and returns a single value

### Common aggregate functions

Aggregate functions will automatically chose the columns for which an aggreagation can be performed on.


In [19]:
df.count()


population    12
fertility     12
continent     12
dtype: int64

In [20]:
df.min()


population     100.7
fertility       1.45
continent     Africa
dtype: object

Careful with this (min) it takes them from different rows. 
You'd need to do your selection more carefully if you needed. 

---------------

Select with conditions to be more specific.

In [ ]:
selection = df["population"]==df["population"].min()

In [22]:
df.loc[selection, :]


,population,fertility,continent
Philippines,100.7,2.98,Asia


In [23]:
df.loc[df["fertility"]==df["fertility"].min(),:]


,population,fertility,continent
Japan,126.6,1.45,Asia


**Some built-in aggregate functions**

* count
* mean
* min
* max
* prod
* median 
* std
* var
* quantile

And the great describe method for fast descriptive statistics

In [24]:
df.describe()


,population,fertility
count,12.000000,12.000000
mean,375.350000,2.437500
std,456.517642,1.200781
min,100.700000,1.450000
25%,139.375000,1.737500
50%,185.550000,2.125000
75%,273.650000,2.567500
max,1376.000000,5.890000


## Section II. Apply a customized set of aggregate functions



**a) With aggregate or apply**

Simple aggregate functions on multiple columns.

In [ ]:
#df[['population', 'fertility']].agg(['sum', 'count', 'mean'])


In [25]:
df[['population', 'fertility']].apply(['sum','mean'])

,population,fertility
sum,4504.20,29.2500
mean,375.35,2.4375


### Note: The following is really flexible!

<a id='aggregate-example'>aggregate-example</a>


You can actually define your own aggregate functions.

In [27]:
def no_outlier(x):
    
    value = x[(x<x.quantile(.9)) & (x>x.quantile(.1))]
    
    return value


# You can call this function with
# no_outlier(df['population'])

The outliers are excluded.

In [ ]:
df[["fertility","population"]].apply(no_outlier)

# or
# df[["fertility","population"]].apply(lambda x:no_outlier(x))

## Section III. Groupby in pandas

Data Aggregation in Python is very closely linked to the ``DataFrame.groupby()`` statement.

- Splitting the data into groups based on some criteria.
- Applying a(n aggregate) function to each group independently.
- Combining the results into a data structure.

## Splitting

In [35]:
df.head(2)

,population,fertility,continent
Bangladesh,161.0,2.12,Asia
Brazil,207.8,1.78,South America


In [36]:
g_continent = df.groupby('continent')

In [38]:
g_continent


#### USEFUL!

In [39]:
g_continent.groups

{'Africa': ['Nigeria'], 'Asia': ['Bangladesh', 'China', 'India', 'Indonesia', 'Japan', 'Pakistan', 'Philippines'], 'Europe': ['Russia'], 'North America': ['Mexico', 'United States'], 'South America': ['Brazil']}

In [46]:
g_continent.groups['Africa']


Index(['Nigeria'], dtype='object')

In [47]:
g_continent.get_group('North America')

,population,fertility,continent
Mexico,127.0,2.13,North America
United States,321.8,1.97,North America


In [52]:
## g_continent.groups

In [70]:
# UU Note: The for loop below is not you do typically, we just do it to look at the inner 
# structure of a grouped object.

In [51]:
for continent, continent_df in g_continent:
    if continent=="Asia":
        print('First print statement gives the key:',continent)
        print('\n Second print statement gives actual sub-df:\n',continent_df)
        print('\n')
    

First print statement gives the key: Asia

 Second print statement gives actual sub-df:
              population  fertility continent
Bangladesh        161.0       2.12      Asia
China            1376.0       1.57      Asia
India            1311.1       2.43      Asia
Indonesia         257.6       2.28      Asia
Japan             126.6       1.45      Asia
Pakistan          188.9       3.04      Asia
Philippines       100.7       2.98      Asia




In [75]:
df.groupby('continent').mean()

,population,fertility
continent,,
Africa,182.200000,5.890000
Asia,503.128571,2.267143
Europe,143.500000,1.610000
North America,224.400000,2.050000
South America,207.800000,1.780000


### Create a dataframe for a single country, use UK.

In [66]:
df_short =  pd.DataFrame([[70,1.5,'Europe']],\
                      columns = ['population',\
                                 'fertility',\
                                 'continent'],\
                       index=['UK'])


In [67]:
df_short

,population,fertility,continent
UK,70,1.5,Europe


Extend your original dataframe by adding df_short to it.

In [68]:
df_appended = df.append(df_short)

/var/folders/vg/s8kxjyf14pvdxlpscsf7lwsw0000gn/T/ipykernel_21943/3805022534.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_appended = df.append(df_short)


In [69]:
df_appended

,population,fertility,continent
Bangladesh,161.0,2.12,Asia
Brazil,207.8,1.78,South America
China,1376.0,1.57,Asia
India,1311.1,2.43,Asia
Indonesia,257.6,2.28,Asia
Japan,126.6,1.45,Asia
Mexico,127.0,2.13,North America
Nigeria,182.2,5.89,Africa
Pakistan,188.9,3.04,Asia
Philippines,100.7,2.98,Asia


In [ ]:
#df_appended['language'] = 'EN'

### Groupby an external dictionary with keys from the index.


In [71]:
language = {'Bangladesh':'BN', 'Brazil':'PT', 'China':'CN',
            'India':'HD', 'Indonesia':'ID', 'Japan':'JP',
            'Mexico':'ES', 'Nigeria':'NG', 'Pakistan':'AR',
            'Philippines':'PP', 'Russia':'RU', 'United States':'EN',
           'UK':'EN'}


In [72]:
g_language = df_appended.groupby(language)

In [74]:
g_language.mean()

,population,fertility
AR,188.9,3.040
BN,161.0,2.120
CN,1376.0,1.570
EN,195.9,1.735
ES,127.0,2.130
HD,1311.1,2.430
ID,257.6,2.280
JP,126.6,1.450
NG,182.2,5.890
PP,100.7,2.980


How can you select the result for English?

In [76]:
g_language.get_group('EN')

,population,fertility,continent
United States,321.8,1.97,North America
UK,70.0,1.50,Europe


In [78]:
g_language.agg({'population':'mean','fertility':'min'})

,population,fertility
AR,188.9,3.04
BN,161.0,2.12
CN,1376.0,1.57
EN,195.9,1.50
ES,127.0,2.13
HD,1311.1,2.43
ID,257.6,2.28
JP,126.6,1.45
NG,182.2,5.89
PP,100.7,2.98


In the above English is the only grouped row. What if we want to apply an aggregate function to the whole data frame after the means of groups are found.

### Grouping by multiple criteria

In [ ]:
#g_double = df_appended.groupby(['continent', language])

In [82]:
df_appended.groupby(['continent', language]).mean()

population  fertility
   continent                           
AR Asia                188.9       3.04
BN Asia                161.0       2.12
CN Asia               1376.0       1.57
EN Europe               70.0       1.50
   North America       321.8       1.97
ES North America       127.0       2.13
HD Asia               1311.1       2.43
ID Asia                257.6       2.28
JP Asia                126.6       1.45
NG Africa              182.2       5.89
PP Asia                100.7       2.98
PT South America       207.8       1.78
RU Europe              143.5       1.61

-------------

# BONUS

## Section IV. Apply vs. Transform


**a) With apply**

Simple method on a groupby object of a column.

In [ ]:
df[['population']].apply(lambda x:  x[(x<x.quantile(.9))])



China and India are not in the data frame anymore. What if we group by the continent first?

In [ ]:
df.groupby('continent')[['population']].apply(lambda x:  x[(x<x.quantile(.9))])


For one thing, more countries has NA's this time but also they are not dropped. This is because the quantiles are calculated per group and there are other elements in the groups that don't get dropped.

**b) With transform and comparison**

In [ ]:
df.groupby('continent').transform(len)

In [ ]:
df.groupby('continent').apply(len)